# PPY — Unique Practice Tasks
**All tasks are original. Every code cell runs correctly.**
Work through each section in order. Try to solve before reading the solution cell below it.

Topics: types · operators · strings · collections · functions · comprehensions · iterators · generators · send() · decorators · OOP · magic methods · MRO · exceptions · context managers · collections module · singledispatch


---
## Section 1 · Types & References

### Task 1-1
You have a registry of server configs stored as a list of dicts.
A colleague does `backup = registry` and then mutates it.
Predict the output, then fix the bug so `registry` is unaffected.

In [ ]:
registry = [{'host': 'db1', 'port': 5432}, {'host': 'web1', 'port': 80}]
backup   = registry          # BUG: same object!

backup.append({'host': 'cache1', 'port': 6379})
backup[0]['port'] = 9999

print('registry length:', len(registry))         # 2 or 3?
print('registry[0][port]:', registry[0]['port'])  # 5432 or 9999?

In [ ]:
# SOLUTION
from copy import deepcopy

registry = [{'host': 'db1', 'port': 5432}, {'host': 'web1', 'port': 80}]
backup   = deepcopy(registry)

backup.append({'host': 'cache1', 'port': 6379})
backup[0]['port'] = 9999

print('registry length:   ', len(registry))       # 2  — unaffected
print('registry[0][port]: ', registry[0]['port'])  # 5432 — unaffected
print('backup length:     ', len(backup))          # 3
print('backup[0][port]:   ', backup[0]['port'])    # 9999

### Task 1-2
What is printed? Explain *why* for each line.

In [ ]:
a = (1, 2, [3, 4])
b = a

b[2].append(5)
print(a)           # ?

try:
    b[0] = 99
except TypeError as e:
    print(e)       # ?

print(a is b)      # ?

In [ ]:
# ANSWERS
a = (1, 2, [3, 4])
b = a

b[2].append(5)
print(a)           # (1, 2, [3, 4, 5])  — inner list IS mutable
try:
    b[0] = 99
except TypeError as e:
    print(e)       # 'tuple' object does not support item assignment
print(a is b)      # True — same object

### Task 1-3
Write `safe_append(lst, val)` that returns a *new* list with `val` appended, leaving the original unchanged.

In [ ]:
def safe_append(lst, val):
    # YOUR CODE
    pass

original = [1, 2, 3]
new_lst  = safe_append(original, 4)
print(original)  # [1, 2, 3]   unchanged
print(new_lst)   # [1, 2, 3, 4]

In [ ]:
# SOLUTION
def safe_append(lst, val):
    new = lst[:]
    new.append(val)
    return new

original = [1, 2, 3]
new_lst  = safe_append(original, 4)
print(original)  # [1, 2, 3]
print(new_lst)   # [1, 2, 3, 4]

---
## Section 2 · Operators

### Task 2-1
Without running: what does each line print?

In [ ]:
print(7 // -2)          # ?
print(-7 // 2)          # ?
print(7 % -2)           # ?
print(-7 % 2)           # ?
print(bool(0.0001))     # ?
print(bool(-0.0))       # ?
print(not not [])       # ?
print(not not [0])      # ?

In [ ]:
# ANSWERS
print(7 // -2)          # -4   (floor toward -inf: 7/-2=-3.5 -> -4)
print(-7 // 2)          # -4   (floor: -7/2=-3.5 -> -4)
print(7 % -2)           # -1   (a-(a//b)*b = 7-(-4*-2) = 7-8 = -1)
print(-7 % 2)           # 1    (-7-(-4*2) = -7+8 = 1)
print(bool(0.0001))     # True  (any non-zero float)
print(bool(-0.0))       # False (-0.0 is falsy)
print(not not [])       # False ([] is falsy)
print(not not [0])      # True  ([0] is non-empty)

### Task 2-2
A config loader uses `or` for defaults. Predict each value, then fix it.

In [ ]:
def load(cfg):
    host    = cfg.get('host')    or 'localhost'
    port    = cfg.get('port')    or 8080
    workers = cfg.get('workers') or 4
    return host, port, workers

# port=0 and workers=0 are valid configs — what gets substituted?
print(load({'host': 'prod', 'port': 0, 'workers': 0}))
# ('prod', 8080, 4)  or  ('prod', 0, 0) ?

In [ ]:
# ANSWER: ('prod', 8080, 4) — 0 is falsy so 'or' replaces it!
# FIX: use .get() with explicit None check
def load_fixed(cfg):
    def get(key, default):
        v = cfg.get(key)
        return default if v is None else v
    return get('host','localhost'), get('port',8080), get('workers',4)

print(load_fixed({'host': 'prod', 'port': 0, 'workers': 0}))
# ('prod', 0, 0)  — correct

### Task 2-3
Write `clamp(val, lo, hi)` without any `if` statement or `min`/`max`.

In [ ]:
def clamp(val, lo, hi):
    # YOUR CODE — no if, no min/max
    pass

print(clamp(5, 0, 10))    # 5
print(clamp(-3, 0, 10))   # 0
print(clamp(15, 0, 10))   # 10

In [ ]:
# SOLUTION: booleans are ints
def clamp(val, lo, hi):
    return lo*(val < lo) + hi*(val > hi) + val*(lo <= val <= hi)

print(clamp(5, 0, 10))    # 5
print(clamp(-3, 0, 10))   # 0
print(clamp(15, 0, 10))   # 10

# Ternary version also fine:
def clamp2(val, lo, hi):
    return lo if val < lo else hi if val > hi else val
print(clamp2(5,0,10), clamp2(-3,0,10), clamp2(15,0,10))

---
## Section 3 · Strings

### Task 3-1
Write `title_case(s)` that capitalises every word *except* articles/prepositions in the middle.

In [ ]:
SMALL = {'a','an','the','in','on','at','of','for','and','but','or'}

def title_case(s):
    words = s.lower().split()
    return ' '.join(
        w if (i > 0 and w in SMALL) else w.capitalize()
        for i, w in enumerate(words)
    )

print(title_case('the lord of the rings'))   # The Lord of the Rings
print(title_case('a tale of two cities'))    # A Tale of Two Cities
print(title_case('gone with the wind'))      # Gone with the Wind

### Task 3-2
Parsing log lines. Extract level, timestamp and message.

In [ ]:
def parse_log(line):
    # lines: '[ERROR] 2024-01-15 14:23:01 - Disk full'
    level = line[1 : line.index(']')]
    rest  = line[line.index(']') + 2:]
    ts, msg = rest.split(' - ', 1)
    return {'level': level.strip(), 'timestamp': ts, 'message': msg}

logs = [
    '[ERROR] 2024-01-15 14:23:01 - Disk full',
    '[INFO]  2024-01-15 14:23:05 - Service restarted',
    '[WARN]  2024-01-15 14:23:10 - High memory - consider scaling',
]

for log in logs:
    p = parse_log(log)
    print(f"{p['level']:5} | {p['timestamp']} | {p['message']}")

### Task 3-3
Write `caesar(text, shift)` that shifts letters, preserving case and non-letters.

In [ ]:
def caesar(text, shift):
    result = []
    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result.append(chr((ord(ch) - base + shift) % 26 + base))
        else:
            result.append(ch)
    return ''.join(result)

msg       = 'Hello, World!'
encrypted = caesar(msg, 13)
decrypted = caesar(encrypted, 13)
print(encrypted)         # Uryyb, Jbeyq!
print(decrypted)         # Hello, World!
print(caesar('xyz', 3))  # abc

### Task 3-4
Write `format_table(headers, rows)` that prints a nicely aligned ASCII table.

In [ ]:
def format_table(headers, rows):
    all_rows = [headers] + rows
    widths   = [max(len(str(r[i])) for r in all_rows) for i in range(len(headers))]
    sep      = '+' + '+'.join('-'*(w+2) for w in widths) + '+'

    def fmt_row(row):
        return '|' + '|'.join(f' {str(v):<{w}} ' for v, w in zip(row, widths)) + '|'

    print(sep)
    print(fmt_row(headers))
    print(sep)
    for row in rows:
        print(fmt_row(row))
    print(sep)

format_table(
    ['Name', 'Score', 'Grade'],
    [['Alice', 95, 'A'], ['Bob', 72, 'C'], ['Charlie', 88, 'B']]
)

---
## Section 4 · Collections

### Task 4-1
Build an inverted index: map each word to the set of doc_ids containing it.

In [ ]:
def build_index(docs):
    index = {}
    for doc_id, text in docs:
        for word in text.lower().split():
            index.setdefault(word, set()).add(doc_id)
    return index

def search(index, query):
    words = query.lower().split()
    sets  = [index.get(w, set()) for w in words]
    return set.intersection(*sets) if sets else set()

docs = [
    (1, 'Python is great'),
    (2, 'Python is easy to learn'),
    (3, 'I love great food'),
]
idx = build_index(docs)
print(idx['python'])            # {1, 2}
print(idx['great'])             # {1, 3}
print(search(idx, 'python great'))  # {1}

### Task 4-2
Write `group_by(lst, key_fn)` that groups items by the result of `key_fn`.

In [ ]:
def group_by(lst, key_fn):
    result = {}
    for item in lst:
        result.setdefault(key_fn(item), []).append(item)
    return result

people = [
    {'name': 'Alice', 'dept': 'Eng'},
    {'name': 'Bob',   'dept': 'HR'},
    {'name': 'Carol', 'dept': 'Eng'},
    {'name': 'Dave',  'dept': 'HR'},
    {'name': 'Eve',   'dept': 'Eng'},
]
by_dept = group_by(people, lambda p: p['dept'])
for dept, members in sorted(by_dept.items()):
    print(f"{dept}: {[m['name'] for m in members]}")

### Task 4-3
Implement `Multiset` (like Counter) with `add`, `remove`, `count`, `__contains__`, `most_common(n)`.

In [ ]:
class Multiset:
    def __init__(self, iterable=None):
        self._data = {}
        if iterable:
            for item in iterable: self.add(item)

    def add(self, item):
        self._data[item] = self._data.get(item, 0) + 1

    def remove(self, item):
        if self._data.get(item, 0) == 0: raise KeyError(item)
        self._data[item] -= 1
        if self._data[item] == 0: del self._data[item]

    def count(self, item):   return self._data.get(item, 0)
    def __contains__(self, item): return item in self._data

    def most_common(self, n=None):
        ranked = sorted(self._data.items(), key=lambda x: -x[1])
        return ranked[:n] if n else ranked

    def __repr__(self): return f'Multiset({self._data})'

ms = Multiset('abracadabra')
print(ms.count('a'))       # 5
print('z' in ms)           # False
print(ms.most_common(3))   # [('a',5),('b',2),('r',2)]
ms.remove('a')
print(ms.count('a'))       # 4

### Task 4-4
Write `flatten(nested, depth=1)` that flattens up to `depth` levels.

In [ ]:
def flatten(nested, depth=1):
    result = []
    for item in nested:
        if isinstance(item, list) and depth > 0:
            result.extend(flatten(item, depth-1))
        else:
            result.append(item)
    return result

print(flatten([1,[2,3],[4,[5,6]]]))           # [1,2,3,4,[5,6]]
print(flatten([1,[2,3],[4,[5,6]]], depth=2))  # [1,2,3,4,5,6]
print(flatten([1,[2,[3,[4]]]], depth=99))     # [1,2,3,4]

---
## Section 5 · Functions & Closures

### Task 5-1
Write `memoize(func)` from scratch (no `functools.lru_cache`).

In [ ]:
def memoize(func):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache
    return wrapper

@memoize
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print([fib(i) for i in range(10)])   # [0,1,1,2,3,5,8,13,21,34]
print(len(fib.cache))                # 10

### Task 5-2
Write `partial_apply(func, *bound_args)` that pre-fills the first N arguments.

In [ ]:
def partial_apply(func, *bound_args):
    def wrapper(*args, **kwargs):
        return func(*bound_args, *args, **kwargs)
    return wrapper

def power(base, exp): return base ** exp

square = partial_apply(power, 2)
cube   = partial_apply(power, 3)
print(square(8))    # 256
print(cube(4))      # 81

log_prefix = partial_apply(print, 'LOG:')
log_prefix('server started')   # LOG: server started

### Task 5-3
Capture-in-loop bug. Predict, then fix two different ways.

In [ ]:
# BUG version — what does this print?
funcs = []
for i in range(5):
    funcs.append(lambda x: x * i)

print([f(2) for f in funcs])   # [0,2,4,6,8] or [8,8,8,8,8]?

In [ ]:
# ANSWER: [8,8,8,8,8] — all lambdas capture same 'i' = 4 (last value)

# Fix 1: default argument
funcs1 = [lambda x, i=i: x * i for i in range(5)]
print([f(2) for f in funcs1])   # [0,2,4,6,8]

# Fix 2: factory closure
def make_fn(i): return lambda x: x * i
funcs2 = [make_fn(i) for i in range(5)]
print([f(2) for f in funcs2])   # [0,2,4,6,8]

### Task 5-4
Write `pipeline(*funcs)` that returns a function applying funcs left-to-right.

In [ ]:
def pipeline(*funcs):
    def apply(value):
        for f in funcs:
            value = f(value)
        return value
    return apply

process = pipeline(
    str.strip,
    str.lower,
    lambda s: s.replace(' ', '_'),
    lambda s: s[:20],
)
print(process('  Hello World  '))                    # hello_world
print(process('  Fast Fourier Transform Analysis ')) # fast_fourier_transfo

---
## Section 6 · Comprehensions

### Task 6-1
One-liners only (no for-loops). Solve each.

In [ ]:
# 1. All primes up to 50
primes = [n for n in range(2,51)
          if all(n%d!=0 for d in range(2,int(n**0.5)+1))]
print(primes)

# 2. Transpose a matrix
matrix    = [[1,2,3],[4,5,6],[7,8,9]]
transposed = [[matrix[r][c] for r in range(len(matrix))]
              for c in range(len(matrix[0]))]
print(transposed)   # [[1,4,7],[2,5,8],[3,6,9]]

# 3. Word frequency dict
sentence = 'to be or not to be that is the question to be'
freq = {w: sentence.split().count(w) for w in set(sentence.split())}
print(sorted(freq.items(), key=lambda x: -x[1])[:3])

# 4. Cartesian product without itertools
A, B = [1,2,3], ['a','b']
print([(x,y) for x in A for y in B])

### Task 6-2
Lazy pipeline using generator expressions — no intermediate lists.

In [ ]:
import math

data = range(1, 1001)

# Keep odd numbers, square them, keep if <10000, take sqrt
result = (
    math.sqrt(sq)
    for sq in (n*n for n in data if n % 2)
    if sq < 10000
)

from itertools import islice
print(list(islice(result, 5)))   # [1.0, 3.0, 5.0, 7.0, 9.0]

---
## Section 7 · Iterators

### Task 7-1
Implement `Countdown(start)` that counts from start down to 0.

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self): return self

    def __next__(self):
        if self.current < 0: raise StopIteration
        val = self.current
        self.current -= 1
        return val

print(list(Countdown(5)))   # [5,4,3,2,1,0]
for n in Countdown(3): print(n, end=' ')
print()

### Task 7-2
Implement `ZipLongest(a, b, fillvalue=None)`.

In [ ]:
class ZipLongest:
    def __init__(self, a, b, fillvalue=None):
        self._a    = iter(a)
        self._b    = iter(b)
        self._fill = fillvalue
        self._done = False

    def __iter__(self): return self

    def __next__(self):
        if self._done: raise StopIteration
        a_done = b_done = False
        a_val = b_val = self._fill
        try: a_val = next(self._a)
        except StopIteration: a_done = True
        try: b_val = next(self._b)
        except StopIteration: b_done = True
        if a_done and b_done:
            self._done = True
            raise StopIteration
        return (a_val, b_val)

print(list(ZipLongest([1,2,3], ['a','b'], fillvalue='-')))
# [(1,'a'),(2,'b'),(3,'-')]

### Task 7-3
Implement `Chunked(iterable, size)` that yields fixed-size chunks.

In [ ]:
class Chunked:
    def __init__(self, iterable, size):
        self._it   = iter(iterable)
        self._size = size
        self._done = False

    def __iter__(self): return self

    def __next__(self):
        if self._done: raise StopIteration
        chunk = []
        for _ in range(self._size):
            try: chunk.append(next(self._it))
            except StopIteration:
                self._done = True
                break
        if not chunk: raise StopIteration
        return chunk

print(list(Chunked(range(10), 3)))  # [[0,1,2],[3,4,5],[6,7,8],[9]]
print(list(Chunked('abcde', 2)))    # [['a','b'],['c','d'],['e']]

---
## Section 8 · Generators

### Task 8-1
Write `sliding_window(iterable, n)` yielding overlapping windows.

In [ ]:
from collections import deque

def sliding_window(iterable, n):
    window = deque(maxlen=n)
    for item in iterable:
        window.append(item)
        if len(window) == n:
            yield tuple(window)

print(list(sliding_window([1,2,3,4,5], 3)))
# [(1,2,3),(2,3,4),(3,4,5)]

data = [10, 12, 9, 11, 14, 13, 15]
avgs = [sum(w)/len(w) for w in sliding_window(data, 3)]
print([round(a,2) for a in avgs])   # [10.33,10.67,11.33,12.67,14.0]

### Task 8-2
Write `merge_sorted(*iterables)` — lazily merges sorted streams using a heap.

In [ ]:
import heapq

def merge_sorted(*iterables):
    heap  = []
    iters = [iter(it) for it in iterables]
    for idx, it in enumerate(iters):
        try:
            heapq.heappush(heap, (next(it), idx, it))
        except StopIteration:
            pass
    while heap:
        val, idx, it = heapq.heappop(heap)
        yield val
        try:
            heapq.heappush(heap, (next(it), idx, it))
        except StopIteration:
            pass

a = [1,4,7,10]
b = [2,5,8]
c = [3,6,9,12]
print(list(merge_sorted(a, b, c)))   # [1,2,3,4,5,6,7,8,9,10,12]

### Task 8-3
Write `infinite_cycle(lst)` and `take_while(pred, gen)`.

In [ ]:
def infinite_cycle(lst):
    while True:
        for item in lst: yield item

def take_while(pred, gen):
    for item in gen:
        if not pred(item): return
        yield item

from itertools import islice
colours = infinite_cycle(['red','green','blue'])
print(list(islice(colours, 7)))
# ['red','green','blue','red','green','blue','red']

nums = infinite_cycle([1,2,3,4,5])
print(list(take_while(lambda x: x < 4, nums)))  # [1,2,3]

---
## Section 9 · Two-Way Generators (send)

### Task 9-1
Write a `moving_average(window)` coroutine keeping a rolling average.

In [ ]:
from collections import deque
import functools

def coroutine(func):
    @functools.wraps(func)
    def start(*args, **kwargs):
        gen = func(*args, **kwargs)
        next(gen)
        return gen
    return start

@coroutine
def moving_average(window):
    buf = deque(maxlen=window)
    avg = None
    while True:
        val = yield avg
        buf.append(val)
        avg = sum(buf) / len(buf)

ma = moving_average(3)
for v in [10, 20, 30, 40, 50]:
    print(f'send({v}) -> avg = {ma.send(v):.2f}')
# send(10)->10.00  send(20)->15.00  send(30)->20.00
# send(40)->30.00  send(50)->40.00

### Task 9-2
Write a `traffic_light` coroutine: RED→GREEN→YELLOW→RED cycling on 'next', reset on 'reset'.

In [ ]:
import functools

def coroutine(func):
    @functools.wraps(func)
    def start(*args, **kwargs):
        g = func(*args, **kwargs)
        next(g)
        return g
    return start

@coroutine
def traffic_light():
    states = ['RED','GREEN','YELLOW']
    i = 0
    while True:
        cmd = yield states[i % len(states)]
        if cmd == 'next':  i += 1
        elif cmd == 'reset': i = 0

light = traffic_light()
print(light.send('next'))    # GREEN
print(light.send('next'))    # YELLOW
print(light.send('next'))    # RED
print(light.send('reset'))   # RED
print(light.send('next'))    # GREEN

### Task 9-3
Write a `logger` coroutine that accumulates messages by level.

In [ ]:
import functools

def coroutine(func):
    @functools.wraps(func)
    def start(*a,**kw):
        g=func(*a,**kw); next(g); return g
    return start

@coroutine
def logger():
    log = {'INFO':[],'WARN':[],'ERROR':[]}
    while True:
        entry = yield log
        if entry:
            level, msg = entry
            log.setdefault(level, []).append(msg)

lg = logger()
lg.send(('INFO',  'Server started'))
lg.send(('WARN',  'High memory'))
lg.send(('ERROR', 'Disk full'))
lg.send(('INFO',  'Backup complete'))
lg.send(('ERROR', 'Network timeout'))
result = lg.send(None)
for level, msgs in result.items():
    if msgs: print(f'{level}: {msgs}')

---
## Section 10 · Decorators

### Task 10-1
Write `@throttle(calls, period)` — max `calls` per `period` seconds, raises RuntimeError when exceeded.

In [ ]:
import functools, time

def throttle(calls, period):
    def decorator(func):
        history = []
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = time.monotonic()
            history[:] = [t for t in history if now - t < period]
            if len(history) >= calls:
                raise RuntimeError(f'Rate limit: max {calls} calls per {period}s')
            history.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@throttle(calls=3, period=1)
def fetch(url): return f'data from {url}'

print(fetch('a.com'))
print(fetch('b.com'))
print(fetch('c.com'))
try:
    print(fetch('d.com'))
except RuntimeError as e:
    print('Throttled:', e)

### Task 10-2
Write `@validate(**type_map)` that type-checks arguments by name.

In [ ]:
import functools, inspect

def validate(**type_map):
    def decorator(func):
        sig = inspect.signature(func)
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            for param, val in bound.arguments.items():
                if param in type_map and not isinstance(val, type_map[param]):
                    raise TypeError(
                        f"'{param}' must be {type_map[param].__name__}, "
                        f"got {type(val).__name__}"
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate(name=str, age=int, score=float)
def register(name, age, score=0.0):
    return f'{name} (age {age}) score={score}'

print(register('Alice', 30, 95.5))
try:
    register('Bob', '25', 80.0)
except TypeError as e:
    print(e)

### Task 10-3
Write `@once` — function runs only once; subsequent calls return cached result.

In [ ]:
import functools

def once(func):
    _result = []
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if not _result:
            _result.append(func(*args, **kwargs))
        return _result[0]
    return wrapper

@once
def expensive_init():
    print('Running expensive init...')
    return {'connection': 'db://prod', 'pool': 10}

r1 = expensive_init()
r2 = expensive_init()   # does NOT re-run
r3 = expensive_init()
print(r1 is r2 is r3)   # True
print(r1)

### Task 10-4
Write `@deprecated(msg)` that emits a `DeprecationWarning` only on the first call.

In [ ]:
import functools, warnings

def deprecated(msg=''):
    def decorator(func):
        warned = []
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if not warned:
                warnings.warn(f'{func.__name__} is deprecated. {msg}',
                              DeprecationWarning, stacklevel=2)
                warned.append(True)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@deprecated('Use new_api() instead.')
def old_api(x): return x * 2

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    print(old_api(5))
    print(old_api(6))
    print(old_api(7))
    print(f'Warnings raised: {len(w)}')   # 1

---
## Section 11 · OOP & Properties

### Task 11-1
Build `Temperature` — Celsius storage with `.fahrenheit` and `.kelvin` as settable properties.

In [ ]:
class Temperature:
    def __init__(self, celsius=0.0):
        self._c = float(celsius)

    @property
    def celsius(self): return self._c

    @celsius.setter
    def celsius(self, v):
        if v < -273.15: raise ValueError(f'Below absolute zero: {v}')
        self._c = float(v)

    @property
    def fahrenheit(self): return self._c * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, v): self.celsius = (v - 32) * 5/9

    @property
    def kelvin(self): return self._c + 273.15

    @kelvin.setter
    def kelvin(self, v): self.celsius = v - 273.15

    def __repr__(self):
        return f'Temperature({self._c:.2f}°C / {self.fahrenheit:.2f}°F / {self.kelvin:.2f}K)'

t = Temperature(100)
print(t)
t.fahrenheit = 32
print(t)
t.kelvin = 0
print(t)
try:
    t.celsius = -300
except ValueError as e:
    print(e)

### Task 11-2
Build `Matrix` supporting `+`, `*` (scalar and matrix), `==`, and pretty `__str__`.

In [ ]:
class Matrix:
    def __init__(self, data):
        self._data = [row[:] for row in data]
        self.rows  = len(data)
        self.cols  = len(data[0])

    def __add__(self, other):
        if (self.rows, self.cols) != (other.rows, other.cols):
            raise ValueError('Shape mismatch')
        return Matrix([[self._data[r][c] + other._data[r][c]
                        for c in range(self.cols)]
                       for r in range(self.rows)])

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Matrix([[v*other for v in row] for row in self._data])
        if self.cols != other.rows: raise ValueError('Shape mismatch for matmul')
        return Matrix([[sum(self._data[r][k]*other._data[k][c]
                            for k in range(self.cols))
                        for c in range(other.cols)]
                       for r in range(self.rows)])

    def __rmul__(self, s): return self.__mul__(s)
    def __eq__(self, other): return self._data == other._data

    def __str__(self):
        w    = max(len(str(v)) for row in self._data for v in row)
        rows = ['[ '+'  '.join(f'{v:{w}}' for v in row)+' ]' for row in self._data]
        return '\n'.join(rows)

A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])
print(A + B); print('---')
print(A * 3); print('---')
print(A * B)
print(A == Matrix([[1,2],[3,4]]))   # True

### Task 11-3
Build `Observable` base class with `subscribe`, `unsubscribe`, `notify`.

In [ ]:
class Observable:
    def __init__(self): self._observers = {}

    def subscribe(self, event, cb):
        self._observers.setdefault(event, []).append(cb)

    def unsubscribe(self, event, cb):
        self._observers.get(event, []).remove(cb)

    def notify(self, event, data=None):
        for cb in self._observers.get(event, []): cb(data)

class StockTicker(Observable):
    def __init__(self, symbol):
        super().__init__()
        self.symbol = symbol
        self._price = 0.0

    @property
    def price(self): return self._price

    @price.setter
    def price(self, value):
        old = self._price
        self._price = value
        self.notify('price_change', {'symbol':self.symbol,'old':old,'new':value})
        if old > 0 and value > old * 1.05:
            self.notify('spike', {'symbol':self.symbol,'pct':(value/old-1)*100})

aapl = StockTicker('AAPL')
aapl.subscribe('price_change', lambda d: print(f"{d['symbol']}: {d['old']} -> {d['new']}"))
aapl.subscribe('spike',        lambda d: print(f"SPIKE! {d['symbol']} +{d['pct']:.1f}%"))

aapl.price = 150.0
aapl.price = 152.0
aapl.price = 160.0   # >5% spike

---
## Section 12 · Magic Methods

### Task 12-1
Build `FrozenCounter` — immutable, hashable Counter. Support `+` between two instances.

In [ ]:
class FrozenCounter:
    def __init__(self, iterable=None):
        counts = {}
        if iterable:
            for item in iterable:
                counts[item] = counts.get(item, 0) + 1
        self._data = tuple(sorted(counts.items()))

    def __getitem__(self, key): return dict(self._data).get(key, 0)

    def __add__(self, other):
        combined = dict(self._data)
        for k, v in other._data:
            combined[k] = combined.get(k, 0) + v
        obj = FrozenCounter.__new__(FrozenCounter)
        obj._data = tuple(sorted(combined.items()))
        return obj

    def __hash__(self):   return hash(self._data)
    def __eq__(self, o):  return self._data == o._data
    def __len__(self):    return sum(v for _,v in self._data)
    def __repr__(self):   return f'FrozenCounter({dict(self._data)})'

a = FrozenCounter('abracadabra')
b = FrozenCounter('banana')
print(a)
print(a['a'])       # 5
print(a + b)
print(hash(a))      # hashable
print(len({a, b}))  # 2 — usable in set

### Task 12-2
Build `SlottedList` — a list that only accepts elements of a given type.

In [ ]:
class SlottedList:
    def __init__(self, dtype, iterable=None):
        self._dtype = dtype
        self._data  = []
        if iterable:
            for item in iterable: self.append(item)

    def _check(self, v):
        if not isinstance(v, self._dtype):
            raise TypeError(f'Expected {self._dtype.__name__}, got {type(v).__name__}')

    def append(self, v):         self._check(v); self._data.append(v)
    def __setitem__(self, i, v): self._check(v); self._data[i] = v
    def __getitem__(self, i):    return self._data[i]
    def __len__(self):           return len(self._data)
    def __iter__(self):          return iter(self._data)
    def __contains__(self, v):   return v in self._data

    def __add__(self, other):
        result = SlottedList(self._dtype, self._data)
        for item in other: result.append(item)
        return result

    def __repr__(self): return f'SlottedList({self._dtype.__name__}, {self._data})'

ints = SlottedList(int, [1,2,3])
ints.append(4)
print(ints)
print(3 in ints)
try:
    ints.append('x')
except TypeError as e:
    print(e)
print(ints + SlottedList(int, [5,6]))

---
## Section 13 · Inheritance & MRO

### Task 13-1
Mixin pattern: `LogMixin`, `ValidationMixin`, `TimestampMixin` — all cooperate via MRO.

In [ ]:
import time

class LogMixin:
    def save(self):
        print(f'[LOG] Saving {self.__class__.__name__}')
        return super().save()

class ValidationMixin:
    def save(self):
        print(f'[VALIDATE] Checking {self.__class__.__name__}')
        if not self.is_valid(): raise ValueError('Validation failed')
        return super().save()
    def is_valid(self): return True

class TimestampMixin:
    def save(self):
        self.updated_at = time.time()
        print('[TIMESTAMP] Set updated_at')
        return super().save()

class Base:
    def save(self):
        print('[BASE] Saving to DB')
        return True

class User(LogMixin, ValidationMixin, TimestampMixin, Base):
    def __init__(self, name): self.name = name
    def is_valid(self): return bool(self.name)

u = User('Alice')
u.save()
print('MRO:', [c.__name__ for c in User.__mro__])

### Task 13-2
Animal hierarchy with polymorphic `sound()` and `describe()`. Show `super()` chain.

In [ ]:
class Animal:
    def __init__(self, name, age):
        self.name = name; self.age = age
    def sound(self): return '...'
    def describe(self):
        return f'{self.__class__.__name__} {self.name}, age {self.age}, says "{self.sound()}"'

class Mammal(Animal):
    def __init__(self, name, age, fur):
        super().__init__(name, age)
        self.fur = fur
    def describe(self):
        return super().describe() + f', fur: {self.fur}'

class Dog(Mammal):
    def sound(self): return 'Woof'

class Cat(Mammal):
    def sound(self): return 'Meow'

class GoldenRetriever(Dog):
    def __init__(self, name, age):
        super().__init__(name, age, fur='golden')

for a in [Dog('Rex',3,'brown'), Cat('Whiskers',5,'orange'), GoldenRetriever('Buddy',2)]:
    print(a.describe())

---
## Section 14 · Exceptions

### Task 14-1
Build a custom exception hierarchy for a payment system.

In [ ]:
class PaymentError(Exception):
    def __init__(self, msg, code=None):
        super().__init__(msg); self.code = code

class InsufficientFundsError(PaymentError):
    def __init__(self, balance, amount):
        super().__init__(f'Cannot charge {amount:.2f}: balance is {balance:.2f}',
                         code='INSUFFICIENT_FUNDS')
        self.deficit = amount - balance

class LimitExceededError(PaymentError):
    def __init__(self, limit, amount):
        super().__init__(f'{amount:.2f} exceeds limit {limit:.2f}',
                         code='LIMIT_EXCEEDED')

def process_payment(balance, amount, limit=500.0):
    if amount > limit: raise LimitExceededError(limit, amount)
    if balance < amount: raise InsufficientFundsError(balance, amount)
    return balance - amount

for balance, amount in [(200,150),(200,300),(200,600)]:
    try:
        nb = process_payment(balance, amount)
        print(f'OK: charged {amount}, new balance {nb}')
    except LimitExceededError as e:
        print(f'[{e.code}] {e}')
    except InsufficientFundsError as e:
        print(f'[{e.code}] {e} | deficit: {e.deficit:.2f}')

### Task 14-2
Write `@retry(times, exceptions, backoff)` with exponential back-off.

In [ ]:
import functools, time

def retry(times=3, exceptions=(Exception,), backoff=0.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = backoff
            for attempt in range(1, times+1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == times: raise
                    print(f'  Attempt {attempt} failed: {e}. Retrying in {delay:.2f}s...')
                    time.sleep(delay)
                    delay = delay * 2 if delay else 0
        return wrapper
    return decorator

count = [0]

@retry(times=4, exceptions=(ConnectionError,), backoff=0)
def flaky():
    count[0] += 1
    if count[0] < 3: raise ConnectionError('timeout')
    return 'connected!'

print(flaky())
print('Attempts:', count[0])

---
## Section 15 · Context Managers

### Task 15-1
Write `Atomic(filename)`: write to temp, rename on success, delete temp on error.

In [ ]:
import os, contextlib

@contextlib.contextmanager
def Atomic(filename):
    tmp = filename + '.tmp'
    try:
        with open(tmp, 'w') as f:
            yield f
        os.replace(tmp, filename)
    except Exception:
        if os.path.exists(tmp): os.remove(tmp)
        raise

with Atomic('/tmp/ppy_atomic.txt') as f:
    f.write('hello atomic world')
print(open('/tmp/ppy_atomic.txt').read())

try:
    with Atomic('/tmp/ppy_atomic2.txt') as f:
        f.write('partial...')
        raise RuntimeError('disk error')
except RuntimeError:
    pass

print('Temp exists:', os.path.exists('/tmp/ppy_atomic2.txt.tmp'))   # False
print('Target exists:', os.path.exists('/tmp/ppy_atomic2.txt'))     # False

### Task 15-2
Write `nested(*managers)` — enters all managers, exits in reverse order.

In [ ]:
import contextlib

@contextlib.contextmanager
def nested(*managers):
    entered = []
    try:
        for mgr in managers:
            entered.append(mgr.__enter__())
        yield entered
    except Exception as exc:
        for mgr in reversed(managers):
            try: mgr.__exit__(type(exc), exc, None)
            except Exception: pass
        raise
    else:
        for mgr in reversed(managers):
            mgr.__exit__(None, None, None)

class Tracked:
    def __init__(self, name): self.name = name
    def __enter__(self):
        print(f'  Enter {self.name}'); return self
    def __exit__(self, *args):
        print(f'  Exit  {self.name}')

with nested(Tracked('A'), Tracked('B'), Tracked('C')) as (a, b, c):
    print(f'  Inside: {a.name}, {b.name}, {c.name}')

---
## Section 16 · Collections Module

### Task 16-1
Use `Counter` to find top-3 bigrams in a text.

In [ ]:
from collections import Counter

def top_bigrams(text, n=3):
    words   = text.lower().split()
    bigrams = zip(words, words[1:])
    return Counter(bigrams).most_common(n)

text = ('to be or not to be that is the question '
        'to be is to do to do is to be do be do be do')
for bigram, count in top_bigrams(text):
    print(f"'{bigram[0]} {bigram[1]}': {count}")

### Task 16-2
Build `FixedSizeHistory(maxlen)` using deque.

In [ ]:
from collections import deque

class FixedSizeHistory:
    def __init__(self, maxlen):
        self._h = deque(maxlen=maxlen)

    def push(self, event): self._h.append(event)
    def last(self, n=1):   return list(self._h)[-n:]
    def oldest(self):      return self._h[0] if self._h else None
    def __len__(self):     return len(self._h)
    def __iter__(self):    return iter(self._h)
    def __repr__(self):    return f'History({list(self._h)})'

hist = FixedSizeHistory(5)
for cmd in ['open','edit 1','edit 2','save','close','reopen','edit 5']:
    hist.push(cmd)

print(hist)
print('Last 2:', hist.last(2))
print('Oldest:', hist.oldest())

### Task 16-3
Use `namedtuple` + `defaultdict` to group contacts by country code.

In [ ]:
from collections import namedtuple, defaultdict

Contact = namedtuple('Contact', ['name','phone','country_code'])

contacts = [
    Contact('Alice',   '+48-111-222', '+48'),
    Contact('Bob',     '+44-777-888', '+44'),
    Contact('Charlie', '+48-555-444', '+48'),
    Contact('Dave',    '+1-212-555',  '+1'),
    Contact('Eve',     '+44-020-794', '+44'),
]

book = defaultdict(list)
for c in contacts: book[c.country_code].append(c)

for code, people in sorted(book.items()):
    print(f'{code}: {[p.name for p in people]}')

---
## Section 17 · singledispatch

### Task 17-1
Write `serialize(obj)` using singledispatch — converts Python objects to JSON-compatible types.

In [ ]:
from functools import singledispatch
from datetime  import date
from decimal   import Decimal
import json

@singledispatch
def serialize(obj):
    raise TypeError(f'Cannot serialize {type(obj).__name__}')

@serialize.register(int)
@serialize.register(float)
@serialize.register(bool)
@serialize.register(type(None))
def _(obj): return obj

@serialize.register(str)
def _(obj): return obj

@serialize.register(list)
@serialize.register(tuple)
def _(obj): return [serialize(item) for item in obj]

@serialize.register(dict)
def _(obj): return {str(k): serialize(v) for k,v in obj.items()}

@serialize.register(date)
def _(obj): return obj.isoformat()

@serialize.register(Decimal)
def _(obj): return float(obj)

data = {
    'name':   'Alice',
    'score':  Decimal('95.5'),
    'joined': date(2024, 1, 15),
    'tags':   ('python','dev'),
    'active': True,
    'address':None,
    'meta':   {'level':3,'badges':[1,2,3]},
}

print(json.dumps(serialize(data), indent=2))

### Task 17-2
Write a `Renderer` class using `singledispatchmethod` to render HTML nodes.

In [ ]:
from functools import singledispatchmethod
from dataclasses import dataclass

@dataclass
class Heading:   text: str; level: int = 1
@dataclass
class Paragraph: text: str
@dataclass
class Link:      text: str; href: str
@dataclass
class CodeBlock: code: str; lang: str = ''
@dataclass
class BulletList: items: list

class Renderer:
    @singledispatchmethod
    def render(self, node):
        raise TypeError(f'No renderer for {type(node).__name__}')

    @render.register(Heading)
    def _(self, node):
        return f'<h{node.level}>{node.text}</h{node.level}>'

    @render.register(Paragraph)
    def _(self, node):
        return f'<p>{node.text}</p>'

    @render.register(Link)
    def _(self, node):
        return f'<a href="{node.href}">{node.text}</a>'

    @render.register(CodeBlock)
    def _(self, node):
        lang = f' class="language-{node.lang}"' if node.lang else ''
        return f'<pre><code{lang}>{node.code}</code></pre>'

    @render.register(BulletList)
    def _(self, node):
        items = ''.join(f'<li>{i}</li>' for i in node.items)
        return f'<ul>{items}</ul>'

r = Renderer()
for node in [
    Heading('Hello World'),
    Paragraph('This is a paragraph.'),
    Link('Click here', 'https://python.org'),
    CodeBlock('print("hi")', lang='python'),
    BulletList(['Item 1','Item 2','Item 3']),
]:
    print(r.render(node))

---
## Final Challenge · Everything Combined

### FC-1: Type-Safe Event Bus
Uses: `singledispatch` dispatch · decorator `@handler` for registration · context manager for temporary handlers · custom exceptions


In [ ]:
import functools, contextlib
from dataclasses import dataclass

@dataclass
class UserCreated:   name: str; email: str
@dataclass
class UserDeleted:   user_id: int
@dataclass
class OrderPlaced:   order_id: int; total: float
@dataclass
class PaymentFailed: order_id: int; reason: str

class EventBusError(Exception): pass

class EventBus:
    def __init__(self): self._handlers = {}

    def subscribe(self, etype, h):
        self._handlers.setdefault(etype, []).append(h)

    def unsubscribe(self, etype, h):
        self._handlers.get(etype, []).remove(h)

    def publish(self, event):
        handlers = self._handlers.get(type(event), [])
        if not handlers:
            raise EventBusError(f'No handlers for {type(event).__name__}')
        for h in handlers: h(event)

    @contextlib.contextmanager
    def temporary(self, etype, h):
        self.subscribe(etype, h)
        try: yield
        finally: self.unsubscribe(etype, h)

    def handler(self, *etypes):
        def decorator(func):
            for et in etypes: self.subscribe(et, func)
            return func
        return decorator

bus = EventBus()

@bus.handler(UserCreated)
def welcome_email(e):
    print(f'[EMAIL] Welcome {e.name} <{e.email}>')

@bus.handler(UserCreated, UserDeleted)
def audit_log(e):
    print(f'[AUDIT] {type(e).__name__}: {e}')

@bus.handler(OrderPlaced)
def process_order(e):
    print(f'[ORDER] Processing order {e.order_id} for \${e.total:.2f}')

@bus.handler(PaymentFailed)
def alert_team(e):
    print(f'[ALERT] Payment failed for order {e.order_id}: {e.reason}')

print('=== Publishing events ===')
bus.publish(UserCreated('Alice','alice@example.com'))
bus.publish(OrderPlaced(42, 99.99))
bus.publish(PaymentFailed(42,'Card expired'))

print('\n=== Temporary debug handler ===')
with bus.temporary(OrderPlaced, lambda e: print(f'[DEBUG] order {e.order_id}')):
    bus.publish(OrderPlaced(43, 199.99))

print('\n=== After temporary handler removed ===')
bus.publish(OrderPlaced(44, 49.99))

try:
    bus.publish(UserDeleted(999))
except EventBusError as e:
    print(f'[ERROR] {e}')

### FC-2: Lazy Evaluation Pipeline
Uses: generators · context manager (timed) · method chaining · closures


In [ ]:
import contextlib, time

class LazyPipeline:
    def __init__(self, source):
        self._source = source
        self._ops    = []
        self.stats   = {}

    def _clone(self, op):
        new = LazyPipeline(self._source)
        new._ops = self._ops + [op]
        return new

    def map(self, f):     return self._clone(('map', f))
    def filter(self, p):  return self._clone(('filter', p))
    def take(self, n):    return self._clone(('take', n))
    def drop(self, n):    return self._clone(('drop', n))
    def flatten(self):    return self._clone(('flatten', None))

    def _run(self):
        def apply_op(gen, op, arg):
            if op == 'map':
                return (arg(x) for x in gen)
            elif op == 'filter':
                return (x for x in gen if arg(x))
            elif op == 'take':
                return (x for _, x in zip(range(arg), gen))
            elif op == 'drop':
                def _drop(g, n):
                    for i, x in enumerate(g):
                        if i >= n: yield x
                return _drop(gen, arg)
            elif op == 'flatten':
                def _flat(g):
                    for item in g: yield from item
                return _flat(gen)
            return gen
        gen = iter(self._source)
        for op, arg in self._ops:
            gen = apply_op(gen, op, arg)
        return gen

    def execute(self): return list(self._run())

    def reduce(self, func, initial=None):
        gen = self._run()
        acc = initial if initial is not None else next(gen)
        for item in gen: acc = func(acc, item)
        return acc

    @contextlib.contextmanager
    def timed(self):
        t0 = time.perf_counter()
        yield self
        self.stats['time'] = time.perf_counter() - t0
        self.stats['ops']  = len(self._ops)

data = range(1, 101)

result = (LazyPipeline(data)
          .filter(lambda x: x % 2 == 0)
          .map(lambda x: x ** 2)
          .take(5)
          .execute())
print('First 5 squares of evens:', result)

total = (LazyPipeline(data)
         .filter(lambda x: x % 3 == 0)
         .reduce(lambda a, b: a + b))
print('Sum of multiples of 3:', total)

nested_data = [[1,2,3],[4,5],[6,7,8,9]]
flat = (LazyPipeline(nested_data)
        .flatten()
        .filter(lambda x: x > 4)
        .execute())
print('Flattened and filtered:', flat)

pipeline = LazyPipeline(data).map(lambda x: x*2).filter(lambda x: x > 10)
with pipeline.timed() as p:
    r = p.take(5).execute()
print('Result:', r)
print('Stats:', pipeline.stats)